# Imports and Setup

In [1]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx
from skewerjax.constants.physical_constants import phys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

INFO:2026-01-11 10:18:04,408:jax._src.xla_bridge:752: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-01-11 10:18:04,408 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-01-11 10:18:04,612 - INFO - SkewerBundle class definition complete


In [2]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

### Define some global variables that will be employed throughout

In [3]:
# Create cosmology object
cosmo_Planck15 = Cosmology(
    H0=phys.H0,
    Omega_r_0=9.1436e-5,  # Radiation density (CMB + neutrinos)
    Omega_m_0=0.3089,
    Omega_k_0=0.0,
    Omega_lambda_0=0.6911,
)

In [4]:
L_box = 1000.0  # Mpc

_spatial_points_x = L_box * jnp.array([ # define spatial points along x corresponding to a single skewer
    [0.0, 0.5, 0.5],
    [0.2, 0.5, 0.5],
    [0.3, 0.5, 0.5],
    [0.5, 0.5, 0.5],
    [1.0, 0.5, 0.5],
])

# Validation of skewers checking different stored attributes

In [5]:
# Define spatial points for multiple skewers
spatial_points = jnp.array([
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x
])

# Initial conditions per skewer
a0_per_skewer = 1.0                                               # scale factor at which ICs are defined, same for all skewers
x0_per_skewer = 0.0                                               # comoving distance at which ICs are defined, same for all skewers
ICs_indices = jnp.array([0, 0, -1, -1])                           # indices along the skewer where ICs are defined (0 if the ICs are defined at the first point, -1 if at the last point of the spatial_points array)
ICs_types = ["emission", "absorption", "emission", "absorption"]  # whether the ICs are defined at emission or absorption

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,                                # Let SkewerBundle compute los_directions automatically
    manual_scale_factors=None,                          # let SkewerBundle compute scale factors automatically
    cosmologies=cosmo_Planck15                          # when the manual scale factors are not provided, cosmologies must be defined
)

2026-01-11 10:18:05,003 - INFO - Initializing SkewerBundle with 4 skewers, 5 points per skewer
2026-01-11 10:18:05,072 - INFO - Using per-skewer ICs_defined_at_index: [ 0  0 -1 -1]
2026-01-11 10:18:05,073 - INFO - Using per-skewer emission/absorption flags
2026-01-11 10:18:05,130 - INFO - Computing los_directions automatically from spatial coordinate differences
2026-01-11 10:18:05,433 - INFO - Computing comoving distances along skewer direction
2026-01-11 10:18:05,610 - INFO - Computing scale factors from cosmology
2026-01-11 10:18:06,559 - INFO - Storing quadri-vectors
2026-01-11 10:18:06,602 - INFO - SkewerBundle initialization complete


In [6]:
print("Number of skewers (N_sk):", skewer_bundle.N_sk)
print("Number of points per skewer (N_ppsk):", skewer_bundle.N_ppsk)
print("Number of cosmologies (N_cosmologies):", cosmo_Planck15.N_cosmologies)

print("ICs defined at indices:", skewer_bundle.ICs_defined_at_index)
print("ICs are defined at absorption (if False, it is defined at emission):", skewer_bundle.is_absorption)
print("Array directions of skewers:", skewer_bundle.array_direction)

Number of skewers (N_sk): 4
Number of points per skewer (N_ppsk): 5
Number of cosmologies (N_cosmologies): 1
ICs defined at indices: [ 0  0 -1 -1]
ICs are defined at absorption (if False, it is defined at emission): [False  True False  True]
Array directions of skewers: [ 1 -1 -1  1]


In [7]:
comoving_positions_from_ICs = skewer_bundle.comoving_positions_from_ICs
print("Comoving positions from ICs:", comoving_positions_from_ICs.shape, " - should be (N_sk, N_ppsk)")
print(comoving_positions_from_ICs)

Comoving positions from ICs: (4, 5)  - should be (N_sk, N_ppsk)
[[    0.   200.   300.   500.  1000.]
 [    0.  -200.  -300.  -500. -1000.]
 [    0.   500.   700.   800.  1000.]
 [    0.  -500.  -700.  -800. -1000.]]


In [8]:
los_directions = skewer_bundle.los_directions
print("Line-of-sight directions:", los_directions.shape, " - should be (N_sk, N_ppsk, 3)")
print(los_directions)

Line-of-sight directions: (4, 5, 3)  - should be (N_sk, N_ppsk, 3)
[[[ 1.  0.  0.]
  [ 1.  0.  0.]
  [ 1.  0.  0.]
  [ 1.  0.  0.]
  [ 1.  0.  0.]]

 [[-1.  0.  0.]
  [-1.  0.  0.]
  [-1.  0.  0.]
  [-1.  0.  0.]
  [-1.  0.  0.]]

 [[-1.  0.  0.]
  [-1.  0.  0.]
  [-1.  0.  0.]
  [-1.  0.  0.]
  [-1.  0.  0.]]

 [[ 1.  0.  0.]
  [ 1.  0.  0.]
  [ 1.  0.  0.]
  [ 1.  0.  0.]
  [ 1.  0.  0.]]]


In [9]:
print("scale_factors:", skewer_bundle.scale_factors.shape, " - should be (N_cosmologies, N_sk, N_ppsk)")
print(skewer_bundle.scale_factors)

scale_factors: (1, 4, 5)  - should be (N_cosmologies, N_sk, N_ppsk)
[[[1.         1.0484515  1.0740788  1.1284539  1.2864715 ]
  [1.         0.95490646 0.93349385 0.8927226  0.80109555]
  [1.2864715  1.2190028  1.1875029  1.1284539  1.        ]
  [0.80109555 0.83614093 0.8544369  0.8927226  1.        ]]]


In [10]:
print("quadri_vectors:", skewer_bundle.quadri_vectors.shape, " - should be (N_cosmologies, N_sk, N_ppsk, 4)")
print(skewer_bundle.quadri_vectors)

quadri_vectors: (1, 4, 5, 4)  - should be (N_cosmologies, N_sk, N_ppsk, 4)
[[[[1.0000000e+00 0.0000000e+00 5.0000000e+02 5.0000000e+02]
   [1.0484515e+00 2.0000000e+02 5.0000000e+02 5.0000000e+02]
   [1.0740788e+00 3.0000000e+02 5.0000000e+02 5.0000000e+02]
   [1.1284539e+00 5.0000000e+02 5.0000000e+02 5.0000000e+02]
   [1.2864715e+00 1.0000000e+03 5.0000000e+02 5.0000000e+02]]

  [[1.0000000e+00 0.0000000e+00 5.0000000e+02 5.0000000e+02]
   [9.5490646e-01 2.0000000e+02 5.0000000e+02 5.0000000e+02]
   [9.3349385e-01 3.0000000e+02 5.0000000e+02 5.0000000e+02]
   [8.9272261e-01 5.0000000e+02 5.0000000e+02 5.0000000e+02]
   [8.0109555e-01 1.0000000e+03 5.0000000e+02 5.0000000e+02]]

  [[1.2864715e+00 0.0000000e+00 5.0000000e+02 5.0000000e+02]
   [1.2190028e+00 2.0000000e+02 5.0000000e+02 5.0000000e+02]
   [1.1875029e+00 3.0000000e+02 5.0000000e+02 5.0000000e+02]
   [1.1284539e+00 5.0000000e+02 5.0000000e+02 5.0000000e+02]
   [1.0000000e+00 1.0000000e+03 5.0000000e+02 5.0000000e+02]]

  [[

### Check if the scale factors are correctly computed independent of invariant coordinate transformations

In [11]:
spatial_points = jnp.array([
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x
])

_manual_a0_for_inverse_trajectory = 0.8010958

# Initial conditions per skewer
a0_per_skewer = jnp.array([1.0, 1.0, _manual_a0_for_inverse_trajectory, _manual_a0_for_inverse_trajectory])
x0_per_skewer = jnp.array([0.0, 0.0, L_box, 0.0])
ICs_indices = jnp.array([0, -1, -1, -1])
ICs_types = ["absorption", "absorption", "emission", "emission"]

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,
    cosmologies=cosmo_Planck15
)

2026-01-11 10:18:06,635 - INFO - Initializing SkewerBundle with 4 skewers, 5 points per skewer
2026-01-11 10:18:06,635 - INFO - Using per-skewer ICs_defined_at_index: [ 0 -1 -1 -1]
2026-01-11 10:18:06,636 - INFO - Using per-skewer emission/absorption flags
2026-01-11 10:18:06,636 - INFO - Computing los_directions automatically from spatial coordinate differences
2026-01-11 10:18:06,641 - INFO - Computing comoving distances along skewer direction
2026-01-11 10:18:06,661 - INFO - Computing scale factors from cosmology
2026-01-11 10:18:07,387 - INFO - Storing quadri-vectors
2026-01-11 10:18:07,388 - INFO - SkewerBundle initialization complete


In [12]:
print("Comoving positions from ICs:\n", skewer_bundle.comoving_positions_from_ICs)
print("ICs_defined_at_index:", skewer_bundle.ICs_defined_at_index)
print("ICs are absorption (if False, emission):", skewer_bundle.is_absorption)
print("Array directions of skewers:", skewer_bundle.array_direction)
print("scale_factors:\n", skewer_bundle.scale_factors)

Comoving positions from ICs:
 [[    0.  -200.  -300.  -500. -1000.]
 [    0.  -500.  -700.  -800. -1000.]
 [ 1000.  1500.  1700.  1800.  2000.]
 [    0.   500.   700.   800.  1000.]]
ICs_defined_at_index: [ 0 -1 -1 -1]
ICs are absorption (if False, emission): [ True  True False False]
Array directions of skewers: [-1  1 -1 -1]
scale_factors:
 [[[1.         0.95490646 0.93349385 0.8927226  0.80109555]
  [0.80109555 0.83614093 0.8544369  0.8927226  1.        ]
  [1.0000005  0.95490634 0.93349427 0.89272386 0.8010958 ]
  [1.0000004  0.9549063  0.93349427 0.89272386 0.8010958 ]]]


### How to flip the quadrivectors to obtain an array ordering following the direction in which each skewer is defined

In [13]:
ii_cosmology = 0
_original = skewer_bundle.quadri_vectors[ii_cosmology]
# Flip the skewers directions where necessary to facilitate integration along the photon path
# Apply mask: where True pick reversed, else keep original. Broadcast mask to match array dimensions
mask = skewer_bundle.array_direction == -1
_reversed = jnp.flip(_original, axis=1) # Reverse across the N_ppsk dimension
quadri_vectors_oriented = jnp.where(mask[:, None, None], _reversed, _original)

print("quadri_vectors_oriented:\n", quadri_vectors_oriented.shape, " - should be (N_sk, N_ppsk, 4)")
print(quadri_vectors_oriented)

quadri_vectors_oriented:
 (4, 5, 4)  - should be (N_sk, N_ppsk, 4)
[[[8.0109555e-01 1.0000000e+03 5.0000000e+02 5.0000000e+02]
  [8.9272261e-01 5.0000000e+02 5.0000000e+02 5.0000000e+02]
  [9.3349385e-01 3.0000000e+02 5.0000000e+02 5.0000000e+02]
  [9.5490646e-01 2.0000000e+02 5.0000000e+02 5.0000000e+02]
  [1.0000000e+00 0.0000000e+00 5.0000000e+02 5.0000000e+02]]

 [[8.0109555e-01 0.0000000e+00 5.0000000e+02 5.0000000e+02]
  [8.3614093e-01 2.0000000e+02 5.0000000e+02 5.0000000e+02]
  [8.5443687e-01 3.0000000e+02 5.0000000e+02 5.0000000e+02]
  [8.9272261e-01 5.0000000e+02 5.0000000e+02 5.0000000e+02]
  [1.0000000e+00 1.0000000e+03 5.0000000e+02 5.0000000e+02]]

 [[8.0109578e-01 1.0000000e+03 5.0000000e+02 5.0000000e+02]
  [8.9272386e-01 5.0000000e+02 5.0000000e+02 5.0000000e+02]
  [9.3349427e-01 3.0000000e+02 5.0000000e+02 5.0000000e+02]
  [9.5490634e-01 2.0000000e+02 5.0000000e+02 5.0000000e+02]
  [1.0000005e+00 0.0000000e+00 5.0000000e+02 5.0000000e+02]]

 [[8.0109578e-01 1.0000000e

### Check if los_directions are correctly computed 

In [14]:
_spatial_points_zigzag = L_box * jnp.array([
    [0.0, 0.0, 0.0],
    [0.1, 0.2, 0.2],
    [0.3, 0.3, 0.3],
    [1.0, 1.0, 1.0],
])

In [15]:
spatial_points = jnp.array([
    _spatial_points_zigzag,
    _spatial_points_zigzag,
    _spatial_points_zigzag,
    _spatial_points_zigzag
])

# Initial conditions per skewer
a0_per_skewer = 1.0
x0_per_skewer = 0.0
ICs_indices = jnp.array([0, 0, -1, -1])
ICs_types = ["emission", "absorption", "emission", "absorption"]

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,
    cosmologies=cosmo_Planck15
)

2026-01-11 10:18:07,520 - INFO - Initializing SkewerBundle with 4 skewers, 4 points per skewer
2026-01-11 10:18:07,521 - INFO - Using per-skewer ICs_defined_at_index: [ 0  0 -1 -1]
2026-01-11 10:18:07,521 - INFO - Using per-skewer emission/absorption flags
2026-01-11 10:18:07,522 - INFO - Computing los_directions automatically from spatial coordinate differences
2026-01-11 10:18:07,707 - INFO - Computing comoving distances along skewer direction
2026-01-11 10:18:07,837 - INFO - Computing scale factors from cosmology
2026-01-11 10:18:08,683 - INFO - Storing quadri-vectors
2026-01-11 10:18:08,726 - INFO - SkewerBundle initialization complete


In [16]:
print("Line-of-sight directions:\n", skewer_bundle.los_directions)

Line-of-sight directions:
 [[[ 0.3333333   0.6666666   0.6666666 ]
  [ 0.81649655  0.40824828  0.40824828]
  [ 0.57735026  0.57735026  0.57735026]
  [ 0.57735026  0.57735026  0.57735026]]

 [[-0.3333333  -0.6666666  -0.6666666 ]
  [-0.3333333  -0.6666666  -0.6666666 ]
  [-0.81649655 -0.40824828 -0.40824828]
  [-0.57735026 -0.57735026 -0.57735026]]

 [[-0.3333333  -0.6666666  -0.6666666 ]
  [-0.3333333  -0.6666666  -0.6666666 ]
  [-0.81649655 -0.40824828 -0.40824828]
  [-0.57735026 -0.57735026 -0.57735026]]

 [[ 0.3333333   0.6666666   0.6666666 ]
  [ 0.81649655  0.40824828  0.40824828]
  [ 0.57735026  0.57735026  0.57735026]
  [ 0.57735026  0.57735026  0.57735026]]]


### Validation of scale factors computation employing several cosmologies

In [17]:
cosmologies = Cosmology(
    H0             = jnp.array([phys.H0, phys.H0, phys.H0, phys.H0, phys.H0]),
    Omega_r_0      = jnp.array([1.0, 0.0, 0.0, 0.0, 9.1436e-5]),
    Omega_m_0      = jnp.array([0.0, 1.0, 0.0, 0.0, 0.3089]),
    Omega_k_0      = jnp.array([0.0, 0.0, 1.0, 0.0, 0.0]),
    Omega_lambda_0 = jnp.array([0.0, 0.0, 0.0, 1.0, 0.6911]),
)

# Define spatial points for multiple skewers
spatial_points = jnp.array([
    _spatial_points_x,
    _spatial_points_x
])

# Initial conditions per skewer
a0_per_skewer = jnp.array([_manual_a0_for_inverse_trajectory, 1.0])
x0_per_skewer = 0.0
ICs_indices = jnp.array([0, 0])
ICs_types = ["emission", "absorption"]

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,
    cosmologies=cosmologies
)

scale_factors = skewer_bundle.scale_factors
# show if dimensions coincide with expectations
print("Number of skewers (N_sk):", skewer_bundle.N_sk)
print("Number of points per skewer (N_ppsk):", skewer_bundle.N_ppsk)
print("Number of cosmologies (N_cosmologies):", cosmologies.N_cosmologies)
print("scale_factors dimensions:\n", scale_factors.shape, " - should be (N_cosmologies, N_sk, N_ppsk)")
print("scale_factors:\n", scale_factors)

2026-01-11 10:18:08,844 - INFO - Initializing SkewerBundle with 2 skewers, 5 points per skewer
2026-01-11 10:18:08,912 - INFO - Using per-skewer ICs_defined_at_index: [0 0]
2026-01-11 10:18:08,913 - INFO - Using per-skewer emission/absorption flags
2026-01-11 10:18:08,970 - INFO - Computing los_directions automatically from spatial coordinate differences
2026-01-11 10:18:09,219 - INFO - Computing comoving distances along skewer direction
2026-01-11 10:18:09,380 - INFO - Computing scale factors from cosmology
2026-01-11 10:18:10,133 - INFO - Storing quadri-vectors
2026-01-11 10:18:10,196 - INFO - SkewerBundle initialization complete


Number of skewers (N_sk): 2
Number of points per skewer (N_ppsk): 5
Number of cosmologies (N_cosmologies): 5
scale_factors dimensions:
 (5, 2, 5)  - should be (N_cosmologies, N_sk, N_ppsk)
scale_factors:
 [[[0.8010958  0.8477948  0.87114424 0.91784316 1.0345906 ]
  [1.         0.9533011  0.92995155 0.8832526  0.7665051 ]]

 [[0.8010958  0.8434384  0.8650186  0.9089968  1.0237129 ]
  [1.         0.9538463  0.9311782  0.8866601  0.7801351 ]]

 [[0.8010958  0.8369428  0.8521814  0.8770063  0.91760665]
  [1.         0.9571762  0.940072   0.91345775 0.8730349 ]]

 [[0.8010958  0.83223045 0.848724   0.8837503  0.98542017]
  [1.         0.9553839  0.93453574 0.8954573  0.8107052 ]]

 [[0.8010958  0.83614075 0.8544373  0.89272386 1.0000004 ]
  [1.         0.95490646 0.93349385 0.8927226  0.80109555]]]
